# 03 --- Signal Detection & Modulation Classification

After channelization, PINGU uses two stages to identify and characterize
signals of interest:

1. **Energy Detection** -- A CFAR (Constant False Alarm Rate) energy
   detector scans each narrowband channel for signals above the noise floor.
2. **Modulation Classification** -- A 1-D CNN (`AMCCNN`) classifies the
   detected signal's modulation type.

This notebook demonstrates both components.

In [ ]:
%matplotlib inline

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

from pingu.detector.energy import EnergyDetector
from pingu.channelizer.polyphase import PolyphaseChannelizer
from pingu.classifier.models.amc_cnn import AMCCNN
from pingu.classifier.inference import AMCInference
from pingu.synthetic.signals import generate_signal
from pingu.synthetic.noise import add_awgn, generate_noise
from pingu.types import ModulationType, ChannelizedFrame

# Reproducible random generator
rng = np.random.default_rng(seed=42)

print("Imports OK")

## Energy Detection

The `EnergyDetector` implements a cell-averaging CFAR algorithm. The input
is a `ChannelizedFrame` whose per-channel time-domain samples are divided
into non-overlapping **blocks** (test cells). For each test cell the
detector estimates the local noise floor from surrounding **reference cells**
(skipping **guard cells** to avoid signal self-masking) and compares the
test-cell energy to a threshold derived from the desired probability of
false alarm (Pfa).

A detection is declared when the test-cell energy exceeds the adaptive
threshold.

In [ ]:
# Build a synthetic channelized frame:
#   8 channels, block_size=128 samples, 40 blocks => 5120 samples per channel
#   Channels 2 and 5 contain injected signals at 15 dB SNR.
#   The remaining 6 channels contain noise only.

N_CHANNELS = 8
BLOCK_SIZE = 128
N_BLOCKS = 40
N_SAMPLES = BLOCK_SIZE * N_BLOCKS  # 5120 samples per channel
SAMPLE_RATE = 3000.0  # per-channel sample rate (Hz)
SIGNAL_SNR_DB = 15.0

# Generate noise-only channels
channels = np.zeros((N_CHANNELS, N_SAMPLES), dtype=np.complex64)
for ch in range(N_CHANNELS):
    channels[ch] = generate_noise(N_SAMPLES, rng=rng)

# Inject signals into channels 2 and 5
signal_channels = [2, 5]
for ch in signal_channels:
    # Generate a BPSK signal as the "target"
    sig = generate_signal(
        ModulationType.BPSK,
        sample_rate=SAMPLE_RATE,
        duration=N_SAMPLES / SAMPLE_RATE,
        rng=rng,
    )
    # Inject: scale signal for desired SNR above noise floor
    # Noise is unit-power, so signal amplitude = sqrt(10^(SNR/10))
    sig_amplitude = np.sqrt(10 ** (SIGNAL_SNR_DB / 10))
    channels[ch] = (channels[ch].astype(np.complex128) + sig_amplitude * sig.astype(np.complex128)).astype(np.complex64)

# Build the ChannelizedFrame
channel_bw = SAMPLE_RATE
center_freq = 14.1e6
channel_freqs = np.array([
    center_freq - (N_CHANNELS / 2 - i - 0.5) * channel_bw
    for i in range(N_CHANNELS)
])

chan_frame = ChannelizedFrame(
    receiver_id="RX-DEMO",
    channels=channels,
    channel_freqs=channel_freqs,
    channel_bw=channel_bw,
    sample_rate=SAMPLE_RATE,
    timestamp=0.0,
)

print(f"ChannelizedFrame: {N_CHANNELS} channels, {N_SAMPLES} samples/channel")
print(f"Signal injected into channels: {signal_channels} at {SIGNAL_SNR_DB} dB SNR")

# Run the energy detector
detector = EnergyDetector(
    pfa=1e-3,
    block_size=BLOCK_SIZE,
    guard_cells=1,
    reference_cells=8,
)

detections = detector.detect(chan_frame)

print(f"\nDetections ({len(detections)} found):")
for det in detections:
    print(f"  Channel {det.channel_index}  freq={det.center_freq/1e6:.4f} MHz  "
          f"SNR={det.snr_estimate:.1f} dB  BW={det.bandwidth:.0f} Hz")

In [ ]:
# Visualize: per-channel energy as a heatmap (channels x blocks)
# Compute per-block energy for every channel
energy_map = np.zeros((N_CHANNELS, N_BLOCKS), dtype=np.float64)
for ch in range(N_CHANNELS):
    for blk in range(N_BLOCKS):
        start = blk * BLOCK_SIZE
        end = start + BLOCK_SIZE
        segment = channels[ch, start:end]
        energy_map[ch, blk] = float(np.sum(np.abs(segment.astype(np.complex128)) ** 2))

energy_map_db = 10 * np.log10(energy_map + 1e-30)

# Identify detected channel indices
detected_channels = {det.channel_index for det in detections}

fig, ax = plt.subplots(figsize=(14, 5))
im = ax.imshow(
    energy_map_db,
    aspect="auto",
    origin="lower",
    cmap="viridis",
    interpolation="nearest",
)
ax.set_xlabel("Block Index")
ax.set_ylabel("Channel Index")
ax.set_yticks(range(N_CHANNELS))
ax.set_yticklabels([f"Ch {i}" for i in range(N_CHANNELS)])
ax.set_title("Per-Channel Energy Heatmap (dB)", fontsize=13, fontweight="bold")
fig.colorbar(im, ax=ax, label="Energy (dB)")

# Mark detected channels with a red rectangle
for ch in detected_channels:
    rect = mpatches.FancyBboxPatch(
        (-0.5, ch - 0.5), N_BLOCKS, 1.0,
        linewidth=2.5, edgecolor="red", facecolor="none",
        boxstyle="round,pad=0.05",
    )
    ax.add_patch(rect)
    ax.text(N_BLOCKS + 0.5, ch, "DETECTED", color="red", fontweight="bold",
            va="center", fontsize=9)

fig.tight_layout()
plt.show()

## CNN Modulation Classifier

The `AMCCNN` is a 1-D convolutional neural network that classifies raw IQ
segments into one of 8 modulation types (SSB, CW, AM, FSK2, FSK4, BPSK,
QPSK, NOISE).

**Architecture overview:**
- Three Conv1d stages: 2 -> 64 -> 128 -> 256 filters (kernel sizes 7, 5, 3).
- Each stage: Conv1d + BatchNorm1d + ReLU + MaxPool1d.
- Global average pooling.
- Fully-connected head: 256 -> 128 -> num_classes.

In [ ]:
import torch

# Instantiate the CNN (untrained)
model = AMCCNN(input_length=1024, num_classes=8)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print("AMCCNN Model Summary")
print("=" * 50)
print(f"Input shape   : (batch, 2, 1024)")
print(f"Output shape  : (batch, 8)")
print(f"Total params  : {total_params:,}")
print(f"Trainable     : {trainable_params:,}")
print()

# Forward pass with random input to verify output shape
dummy_input = torch.randn(4, 2, 1024)  # batch of 4
with torch.no_grad():
    output = model(dummy_input)

print(f"Forward pass:")
print(f"  Input  shape: {tuple(dummy_input.shape)}")
print(f"  Output shape: {tuple(output.shape)}")
print(f"  Output (first sample logits): {output[0].numpy().round(3)}")

In [ ]:
# Use AMCInference (untrained -- no checkpoint) to classify synthetic signals
inference = AMCInference(checkpoint_path=None, input_length=1024, device="cpu")

# Generate several synthetic signals for classification
test_mods = [
    ModulationType.CW,
    ModulationType.AM,
    ModulationType.BPSK,
    ModulationType.QPSK,
    ModulationType.FSK2,
]

print(f"{'True Modulation':>18s}  {'Predicted':>12s}  {'Confidence':>12s}")
print("-" * 48)

for mod in test_mods:
    # Generate a signal with enough samples for the classifier
    sig = generate_signal(
        modulation=mod,
        sample_rate=48_000,
        duration=1024 / 48_000,  # exactly 1024 samples
        rng=rng,
    )
    predicted_mod, confidence = inference.classify(sig)
    match = "*" if predicted_mod == mod else " "
    print(f"{mod.name:>18s}  {predicted_mod.name:>12s}  {confidence:>11.4f}  {match}")

print()
print("NOTE: The model is untrained, so predictions are effectively random.")
print("      After training with `scripts/train_amc.py`, accuracy improves substantially.")

## Training the Classifier

The AMC CNN is trained on synthetic data generated by the signal generators
in `pingu.synthetic.signals`. The training script uses PyTorch Lightning
and can be launched as follows:

```bash
python scripts/train_amc.py \
    --max_epochs 50 \
    --batch_size 256 \
    --learning_rate 1e-3 \
    --input_length 1024 \
    --snr_range -5 20
```

The training loop generates random IQ segments at varying SNR levels,
applies data augmentation (frequency/phase offsets), and optimises the
cross-entropy loss. After training, the best checkpoint is saved and can be
loaded by `AMCInference(checkpoint_path="path/to/best.ckpt")`.

## Summary

- The **EnergyDetector** uses a CFAR algorithm to maintain a controlled
  probability of false alarm (Pfa) while adapting to varying noise levels.
  Guard and reference cells provide robustness against signal self-masking.
- The **AMCCNN** classifier handles modulation recognition after detection.
  It takes raw IQ segments as input and outputs a `ModulationType` with a
  confidence score.
- The untrained model produces random predictions; training on synthetic
  data (via `scripts/train_amc.py`) is required for operational accuracy.